# 06 · Evaluation
**Full LPS evaluation pipeline: ROUGE, BERTScore, Topic Coherence, Persona Fidelity**

Runs LPS-Clustering vs. baselines (Zero-Shot, Top-K RAG) on the test cohort
and reproduces the metrics from **Table 1** of the paper.


In [ ]:

import sys
sys.path.append("../src")

import numpy as np
import json, yaml, pickle
from tqdm import tqdm
from embeddings import UserPrioritizedEmbedder
from clustering import ELCFramework
from retrieval import TwoStageRetriever
from generation import DualStreamGenerator
from evaluation import LPSEvaluator
import anthropic

with open("../configs/default.yaml") as f:
    cfg = yaml.safe_load(f)
with open("../data/cohort.json") as f:
    cohort = json.load(f)
with open("../data/embeddings.pkl", "rb") as f:
    all_embeddings = pickle.load(f)

client   = anthropic.Anthropic()
embedder = UserPrioritizedEmbedder(alpha=cfg["embedding"]["alpha"])
evaluator= LPSEvaluator()
generator= DualStreamGenerator(client=client)
retriever= TwoStageRetriever(encoder=embedder.encoder,
                              top_k_exemplars=cfg["retrieval"]["top_k_exemplars"])

print(f"Evaluating on {len(cohort)} users × {cfg['data']['holdout_conversations']} test queries each")


In [ ]:

# ── Run LPS-Clustering on first 5 users (demo) ───────────────────────
results = {"lps_clustering": [], "zero_shot": [], "topk_rag": []}
pf_scores = {"lps_clustering": [], "zero_shot": [], "topk_rag": []}

DEMO_USERS = list(cohort.keys())[:5]

for uid in tqdm(DEMO_USERS, desc="Users"):
    data = all_embeddings[uid]

    # Annotate kappa
    for inter in data["interactions"]:
        emb = embedder.encode([inter["user"]])[0]
        inter["kappa"] = retriever._infer_query_tone(emb)

    # Fit ELC
    elc = ELCFramework(lambda_decay=cfg["elc"]["lambda_decay"],
                       min_cluster_size=cfg["elc"]["min_cluster_size"],
                       min_samples=cfg["elc"]["min_samples"])
    elc.fit(data["embeddings"], data["timestamps"])

    for conv in cohort[uid]["test"]:
        for pair in conv["pairs"]:
            query = pair["user"]
            ground_truth = pair["model"]  # Use user's actual next turn if available
            query_emb = embedder.encode([query])[0]

            # LPS-Clustering
            tier1 = elc.retrieve_exemplars(query_emb, data["interactions"],
                       top_k_anchors=cfg["retrieval"]["top_k_anchors"],
                       top_k_exemplars=cfg["retrieval"]["top_k_exemplars"] * 2)
            ret_result = retriever.retrieve(query, query_emb, tier1)
            context    = TwoStageRetriever.format_context(ret_result)
            lps_resp   = generator.generate(query, ret_result, context)

            # Zero-Shot baseline
            zs_resp = client.messages.create(
                model="claude-sonnet-4-6", max_tokens=500,
                messages=[{"role": "user", "content": query}]
            ).content[0].text

            results["lps_clustering"].append((lps_resp, ground_truth, query_emb))
            results["zero_shot"].append((zs_resp, ground_truth, query_emb))

print(f"\nGenerated {len(results['lps_clustering'])} response pairs")


In [ ]:

# ── Compute metrics ───────────────────────────────────────────────────
def compute_metrics(pairs, evaluator, embedder, elc):
    preds = [p for p, _, _ in pairs]
    refs  = [r for _, r, _ in pairs]
    pred_embs = embedder.encode(preds)
    # Use the query embedding's closest anchor as TC reference
    anchor_embs = np.stack([
        elc.retrieve_anchors(q_emb, top_k=1)[0].centroid
        for _, _, q_emb in pairs
    ])
    rouge_scores = evaluator.rouge(preds, refs)
    bs = evaluator.bertscore(preds, refs)
    tc = evaluator.topic_coherence(pred_embs, anchor_embs)
    return {
        "ROUGE-1": round(rouge_scores["rouge1"] * 100, 2),
        "ROUGE-L": round(rouge_scores["rougeL"] * 100, 2),
        "BERTScore-F1": round(bs, 4),
        "Topic Coherence": round(tc, 4),
    }

metrics_lps = compute_metrics(results["lps_clustering"], evaluator, embedder, elc)
metrics_zs  = compute_metrics(results["zero_shot"], evaluator, embedder, elc)

import pandas as pd
comparison = pd.DataFrame({
    "Zero-Shot":     metrics_zs,
    "LPS-Clustering": metrics_lps,
}).T
print("\n=== Results (Demo — 5 users) ===")
print(comparison.to_string())
print("\nNote: Full paper results use 4,200 users with 3 random seeds.")
